# DX 603: Project Milestone Two: Modeling and Feature Engineering

### Due: Sunday July 26 @ 11:59PM (with grace period of 2 hours & 1 minute)

### Overview

In Milestone 1, you explored the Zillow dataset, cleaned the data, and developed hypotheses about how preprocessing and feature engineering might improve predictive performance.

In this milestone, you will  develop, evaluate, and refine several machine learning models using those ideas. Rather than simply searching for the best algorithm, you will follow an iterative modeling workflow by:

1. Establishing baseline performance using several regression models.
2. Testing the preprocessing and feature engineering ideas proposed in Milestone 1.
3. Refining the feature set through feature selection.
4. Optimizing model performance through hyperparameter tuning.
5. Comparing the evolution of your models and selecting a final model to evaluate on the held-out test set.

Throughout this milestone, use **repeated 5-fold cross-validation (5 repeats)** to guide your modeling decisions. The held-out test set should be used only once, after all modeling decisions have been completed.




In [1]:
%pip install numpy pandas seaborn matplotlib scikit-learn tqdm requests


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
# ===================================
# Useful Imports: Add more as needed
# ===================================

# Standard Libraries
import os
import time
import math
import io
import zipfile
import requests
from urllib.parse import urlparse
from itertools import chain, combinations

# Data Science Libraries
import numpy as np
import pandas as pd
import seaborn as sns

# Visualization
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.ticker as mticker  # Optional: Format y-axis labels as dollars
import seaborn as sns

# Scikit-learn (Machine Learning)
from sklearn.model_selection import (
    train_test_split, 
    cross_val_score, 
    GridSearchCV, 
    RandomizedSearchCV, 
    RepeatedKFold
)
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error
from sklearn.feature_selection import SequentialFeatureSelector, f_regression, SelectKBest
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor, HistGradientBoostingRegressor

# Progress Tracking

from tqdm import tqdm

# =============================
# Global Variables
# =============================
random_state = 42

# =============================
# Utility Functions
# =============================

# Format y-axis labels as dollars with commas (optional)
def dollar_format(x, pos):
    return f'${x:,.0f}'

# Convert seconds to HH:MM:SS format
def format_hms(seconds):
    return time.strftime("%H:%M:%S", time.gmtime(seconds))



## Prelude: Load Your Preprocessed Dataset from Milestone 1

In Milestone 1, you cleaned the Zillow dataset by removing unsuitable features, handling missing values, and encoding categorical variables. In this milestone, you will build, compare, and improve several regression models using that prepared dataset.

Begin by returning to your Milestone 1 notebook and rerunning your code through Part 3, where your dataset has been completely cleaned and encoded, but before any experimental feature engineering ideas were evaluated. Save these datasets to use as the starting point for this milestone.

For example, do this at the end of Milestone 1:

```python
X_train.to_csv("X_train.csv", index=False)               # or whatever names you gave these sets
X_test.to_csv("X_test.csv", index=False)
y_train.to_csv("y_train.csv", index=False)
y_test.to_csv("y_test.csv", index=False)
```

Then load them at the beginning of the Milestone 2 notebook:

```python
X_train = pd.read_csv("X_train.csv")
X_test = pd.read_csv("X_test.csv")

y_train = pd.read_csv("y_train.csv").squeeze("columns")
y_test = pd.read_csv("y_test.csv").squeeze("columns")
```
#### Feature Scaling

Some regression models, such as **Ridge Regression** and **Lasso Regression**, require feature scaling. If you use one of these models, standardize the predictor variables **using only the training data**, then apply the same transformation to the test data.

```python
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)
```

**Notes**

- Ordinary Linear Regression, Decision Trees, Random Forests, and HistGradientBoosting do **not** require feature scaling.
- If you create additional features later in this milestone and are using a scaled model, repeat the scaling step so the new features are transformed consistently.
- Throughout this milestone, use the same training/test split so that all models are evaluated on identical data.

In [3]:
# Add as many cells as you need
X_train = pd.read_csv("X_train.csv")
X_test = pd.read_csv("X_test.csv")

y_train = pd.read_csv("y_train.csv").squeeze("columns")
y_test = pd.read_csv("y_test.csv").squeeze("columns")

## Problem 1: Model Selection and Baselines [6 pts]

### 1.A Coding

Select **three** regression models from the following list and evaluate each one using the cleaned training dataset.

Use the default hyperparameters provided by scikit-learn (except where scaling is required).

Available models:

* Linear Regression
* Ridge Regression
* Lasso Regression
* Decision Tree Regressor
* Bagging Regressor
* Random Forest Regressor
* HistGradientBoostingRegressor

For each of the three models you choose:

* Train using the **training dataset only**.
* Use **Repeated 5-Fold Cross-Validation** (5 repeats).
* Report validation performance:

  * Mean CV MAE
  * Standard Deviation of CV MAE

In [4]:
# Add as many code cells as needed.
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, RepeatedKFold
from sklearn.tree import DecisionTreeRegressor

In [5]:
#Repeated 5-Fold Cross Validation object
random_state = 42

repeated_cv = RepeatedKFold(
    n_splits=5,
    n_repeats=5,
    random_state=random_state
)

In [6]:
#Defining the three regression models
models = {

    "Linear Regression":
        LinearRegression(),

    "Ridge Regression":
        Pipeline([
            ("scaler", StandardScaler()),
            ("model", Ridge())
        ]),

    "Random Forest":
        RandomForestRegressor(
            random_state=random_state
        )

}

In [7]:
print(X_train.shape)
print(X_test.shape)

(62090, 119)
(15523, 119)


In [8]:
print(type(X_train))
print(X_train.shape)

print(type(y_train))
print(y_train.shape)

<class 'pandas.DataFrame'>
(62090, 119)
<class 'pandas.Series'>
(62090,)


In [9]:
print("Missing values in y_train:", y_train.isna().sum())
print("Total values in y_train:", len(y_train))

Missing values in y_train: 29
Total values in y_train: 62090


In [10]:
# To Keep only rows where y_train is not missing
valid_rows = y_train.notna()

X_train = X_train.loc[valid_rows].copy()
y_train = y_train.loc[valid_rows].copy()

# Reseting indexes so they remain clean and aligned
X_train = X_train.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)

print("Updated X_train shape:", X_train.shape)
print("Updated y_train shape:", y_train.shape)
print("Missing values in y_train:", y_train.isna().sum())
print("Rows match:", len(X_train) == len(y_train))

Updated X_train shape: (62061, 119)
Updated y_train shape: (62061,)
Missing values in y_train: 0
Rows match: True


In [ ]:
#Evaluating each model
results = []

for name, model in models.items():

    print(f"Evaluating {name}...")

    mae_scores = -cross_val_score(

        model,

        X_train,

        y_train,

        scoring="neg_mean_absolute_error",

        cv=repeated_cv,

        n_jobs=-1

    )

    results.append({

        "Model": name,

        "Mean CV MAE": mae_scores.mean(),

        "Std CV MAE": mae_scores.std()

    })

Evaluating Linear Regression...
Evaluating Ridge Regression...
Evaluating Random Forest...


In [ ]:
#Displaying the results
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="Mean CV MAE",
    ascending=True
).reset_index(drop=True)

results_df["Mean CV MAE"] = results_df["Mean CV MAE"].round(2)
results_df["Std CV MAE"] = results_df["Std CV MAE"].round(2)

results_df

,Model,Mean CV MAE,Std CV MAE
0,Random Forest,193121.28,3610.39
1,Ridge Regression,250679.52,2830.30
2,Linear Regression,311679.62,4377.28


### 1.B Discussion

Answer the following questions.

#### 1.B.1

Which of your three models achieved the **lowest validation MAE score **?

> The Random Forest Regressor achieved the lowest validation MAE score, with a Mean CV MAE of 193,121.28. This means it produced the most accurate predictions of the three baseline models.

#### 1.B.2

Which model produced the **smallest standard deviation** across the repeated cross-validation runs? What does this suggest about its stability?

> The Ridge Regression model produced the smallest standard deviation across the repeated cross-validation runs, with a standard deviation of 2,830.30. This suggests that its performance was the most consistent across the different training and validation splits. Although Ridge Regression was not the most accurate model, its lower variation indicates that it produced more stable and reliable results than the other two models.

#### 1.B.3

Did any model appear to overfit or underfit? Explain your reasoning using the training and cross-validation results.

> Based on the cross-validation results alone, I cannot say that any of the models are overfitting because I did not compare their training and validation errors. However, Linear Regression appears to be underfitting since it had the highest Mean CV MAE of the three models. This suggests that the model is too simple to capture the relationships between the features and house values. On the other hand, Random Forest had the lowest Mean CV MAE, which indicates that it fit the data much better. To determine whether any model is actually overfitting, I would need to compare its training performance with its validation or test performance after tuning the models later in the project.

#### 1.B.4

Compare the overall strengths and weaknesses of the three models. Did any model consistently perform better, or were there important tradeoffs between accuracy and stability?

> Each model had its own strengths and weaknesses. Random Forest performed the best overall because it had the lowest Mean CV MAE, making it the most accurate model. Ridge Regression was not as accurate, but it had the smallest standard deviation, meaning its performance was the most consistent across the cross-validation runs. Linear Regression performed the worst, with the highest Mean CV MAE and the largest standard deviation, suggesting it was both less accurate and less stable. Overall, Random Forest provided the best balance between accuracy and stability, while Ridge Regression offered the most consistent results but with lower predictive accuracy.

## Part 2: Evaluate Your Feature Engineering Hypotheses [6 pts]

### 2.A Coding

In **Milestone 1**, you proposed several preprocessing and feature engineering ideas that you believed might improve predictive performance.

Select **at least three** of those ideas and evaluate them.

These may include, for example:

* Creating new features
* Transforming existing features
* Removing features
* Combining features
* Other preprocessing ideas that you proposed in Milestone 1

For each idea:

* Apply the preprocessing or feature engineering to the **training dataset only**.
* Retrain the same three baseline models from **Problem 1** using repeated 5-fold cross-validation (5 repeats). 
* Compare the validation performance (mean CV MAE) and stability (standard deviation of CV MAE) with your original baseline results


> One of the most important things you can learn is that **not every clever idea results in an improvement**--they have to be evaluated by careful experiment.  And negative results are valuable if they are carefully evaluated and discussed!

In [ ]:
#Log-transform skewed features

X_train_log = X_train.copy()

X_train_log["calculatedfinishedsquarefeet"] = np.log1p(
    X_train_log["calculatedfinishedsquarefeet"]
)

In [ ]:
#evaluating the same three models.
results_log = []

for name, model in models.items():

    mae_scores = -cross_val_score(
        model,
        X_train_log,
        y_train,
        scoring="neg_mean_absolute_error",
        cv=repeated_cv,
        n_jobs=-1
    )

    results_log.append({

        "Model": name,
        "Mean CV MAE": mae_scores.mean(),
        "Std CV MAE": mae_scores.std()

    })

results_log = pd.DataFrame(results_log)
results_log

/usr/local/python/3.12.1/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


In [ ]:
#Removing highly correlated features
high_corr_features = [
    "finishedsquarefeet12",
    "calculatedbathnbr"
]

In [ ]:
X_train_corr = X_train.drop(
    columns=high_corr_features,
    errors="ignore"
)

In [ ]:
#Evaluating again
results_corr = []

for name, model in models.items():

    mae_scores = -cross_val_score(
        model,
        X_train_corr,
        y_train,
        scoring="neg_mean_absolute_error",
        cv=repeated_cv,
        n_jobs=-1
    )

    results_corr.append({

        "Model": name,
        "Mean CV MAE": mae_scores.mean(),
        "Std CV MAE": mae_scores.std()

    })

results_corr = pd.DataFrame(results_corr)
results_corr

,Model,Mean CV MAE,Std CV MAE
0,Linear Regression,311679.620652,4377.282121
1,Ridge Regression,250031.997666,2994.649455
2,Random Forest,193497.630783,3661.677063


In [ ]:
#Removing outlier: extreme values using the IQR rule
Q1 = X_train["calculatedfinishedsquarefeet"].quantile(0.25)
Q3 = X_train["calculatedfinishedsquarefeet"].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

mask = (
    (X_train["calculatedfinishedsquarefeet"] >= lower)
    &
    (X_train["calculatedfinishedsquarefeet"] <= upper)
)

X_train_outliers = X_train.loc[mask]
y_train_outliers = y_train.loc[mask]

In [ ]:
#Evaluating again
results_outliers = []

for name, model in models.items():

    mae_scores = -cross_val_score(
        model,
        X_train_outliers,
        y_train_outliers,
        scoring="neg_mean_absolute_error",
        cv=repeated_cv,
        n_jobs=-1
    )

    results_outliers.append({

        "Model": name,
        "Mean CV MAE": mae_scores.mean(),
        "Std CV MAE": mae_scores.std()

    })

results_outliers = pd.DataFrame(results_outliers)
results_outliers

/usr/local/python/3.12.1/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


,Model,Mean CV MAE,Std CV MAE
0,Linear Regression,237867.865504,2286.235195
1,Ridge Regression,199528.675201,1896.954297
2,Random Forest,167480.503734,1680.853702


### 2.B Discussion

Answer the following questions.

#### 2.B.1

Which of your feature engineering ideas produced the largest improvement in validation performance?

> Removing the outliers from calculatedfinishedsquarefeet produced the biggest improvement in validation performance. All three models had lower Mean CV MAE scores after removing the outliers, meaning they made more accurate predictions than the baseline models. The biggest improvement was seen with Linear Regression, but Random Forest still had the lowest Mean CV MAE overall, making it the best-performing model. This suggests that the extreme values were negatively affecting the models, and removing them helped improve their performance.

#### 2.B.2

Were any of your ideas unsuccessful or did they reduce model performance? Briefly explain.

> Yes. Not all of the feature engineering ideas improved the models. The log transformation and removing highly correlated features either resulted in only small improvements or slightly reduced the performance of some models compared to the baseline. This shows that not every feature engineering idea leads to better results. In contrast, removing the outliers produced the most noticeable improvement, lowering the Mean CV MAE for all three models.

#### 2.B.3

Did some models benefit more from feature engineering than others? If so, why do you think this occurred?

> Yes. Some models benefited more from feature engineering than others. Linear Regression showed the biggest improvement in Mean CV MAE after the feature engineering steps, while Random Forest also improved but by a smaller amount. I think this happened because Linear Regression is a simpler model and is more sensitive to issues like outliers and noisy data. Random Forest was already performing well on the original dataset, so there was less room for improvement. Overall, the feature engineering helped all three models, but it had the biggest impact on the simpler models.

#### 2.B.4

Which preprocessing or feature engineering changes will you keep for the remainder of the milestone? Briefly justify your decision.

> For the rest of the milestone, I will keep the outlier removal step because it produced the best overall results. After removing the outliers, all three models achieved lower Mean CV MAE scores and became more consistent across the cross-validation runs. Since this feature engineering idea improved both accuracy and stability, it appears to be the most effective preprocessing step to carry forward into the next part of the project..

## Part 3: Refine the Feature Set [6 pts]

### 3.A Coding

Using your dataset after completing **Part 2** (including any preprocessing and feature engineering changes you decided to keep):

Investigate whether **feature selection** can further improve model performance.

You may use one or more of the following methods:

* Forward Selection (for linear regression models)
* Backward Selection (for linear regression models)
* Feature importance from tree-based models (for decision trees, Random Forests, Bagging, and HistGradientBoosting)
* Another reasonable feature selection method

For each of your three models:

* Select a subset of features using an appropriate feature selection method.
* Retrain the model using only the selected features.
* Evaluate the model using the same repeated cross-validation procedure as before.
* Report the validation performance (the mean and standard deviation of the CV MAE).

> Not every model will necessarily benefit from feature selection. Choose methods that are appropriate for the models you selected. Negative results are valuable if they are carefully evaluated and discussed!

In [ ]:
from sklearn.feature_selection import (
    SequentialFeatureSelector,
    f_regression,
    SelectKBest,
    SelectFromModel
)

from sklearn.pipeline import Pipeline

In [ ]:
#Selecting features for Linear Regression
selector_lr = SelectFromModel(
    LinearRegression(),
    threshold="mean"
)

X_lr = selector_lr.fit_transform(
    X_train_outliers,
    y_train_outliers
)

In [ ]:
#Selecting features for Ridge
selector_ridge = SelectFromModel(
    Ridge(),
    threshold="mean"
)

X_ridge = selector_ridge.fit_transform(
    X_train_outliers,
    y_train_outliers
)

/usr/local/python/3.12.1/lib/python3.12/site-packages/sklearn/linear_model/_ridge.py:227: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 4.9067255577954164e-30.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T


In [ ]:
#Selecting features for Random Forest

selector_rf = SelectFromModel(
    RandomForestRegressor(
        random_state=42
    ),
    threshold="mean"
)

X_rf = selector_rf.fit_transform(
    X_train_outliers,
    y_train_outliers
)

In [ ]:
#Evaluating Linear Regression model
mae = -cross_val_score(
    LinearRegression(),
    X_lr,
    y_train_outliers,
    cv=repeated_cv,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)

print("Linear Regression")
print("Mean:", mae.mean())
print("Std :", mae.std())

Linear Regression
Mean: 237803.92286768998
Std : 2284.7357138325074


In [ ]:
#Evaluating Ridge Regression model
ridge_model = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge())
])

mae = -cross_val_score(
    ridge_model,
    X_ridge,
    y_train_outliers,
    cv=repeated_cv,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)

print("Ridge Regression")
print("Mean:", mae.mean())
print("Std :", mae.std())

Ridge Regression
Mean: 234009.68085816526
Std : 2271.232495392592


In [ ]:
#Evaluating Random Forest model
mae = -cross_val_score(
    RandomForestRegressor(random_state=42),
    X_rf,
    y_train_outliers,
    cv=repeated_cv,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)

print("Random Forest")
print("Mean:", mae.mean())
print("Std :", mae.std())

Random Forest
Mean: 168442.56695773438
Std : 1821.5753536093475


### 3.B Discussion

#### 3.B.1

Did feature selection improve the validation performance of any of your models?

> Feature selection did not really improve the validation performance of my models. Linear Regression showed a very small improvement, but the difference was too small to be meaningful. Ridge Regression and Random Forest both performed worse after feature selection. Based on these results, I don't think reducing the number of features helped improve the models.

#### 3.B.2

Were there features that were consistently retained (or consistently removed) across multiple models?

> I did not notice any features that were consistently retained or removed across all three models. Each model selected features differently based on how it determined feature importance. This shows that the features that are most useful can vary depending on the model being used.

#### 3.B.3

Were any of your engineered features selected as important? If so, what does this suggest about the hypotheses you developed in Milestone 1?

> The feature engineering idea that had the biggest impact was removing the outliers. Although I did not create any new features that were selected during feature selection, removing the outliers improved the performance of all three models before feature selection. This supports the hypothesis I made in Milestone 1 that reducing the effect of extreme values would help the models make more accurate predictions..

#### 3.B.4

After feature selection, did simpler models perform as well as—or better than—the models using the full feature set? Briefly discuss any tradeoffs you observed between model complexity and predictive performance.

> After feature selection, the simpler models did not perform better than the models using the full feature set. Linear Regression showed only a very small improvement, while Ridge Regression and Random Forest both had higher Mean CV MAE scores after feature selection. This suggests that reducing the number of features made the models simpler, but it also removed information that helped improve prediction accuracy. In this case, keeping the full feature set provided better overall performance.

## Part 4: Tune Your Models [8 pts]

### 4.A Coding

Using the three models developed in **Part 3** (including your final preprocessing, feature engineering, and feature selection decisions):

Investigate whether **hyperparameter tuning** can further improve model performance.

For each of your three models:

* Select one or more important hyperparameters to tune.
* Use one or more appropriate tuning methods. Consider first using validation curves (`sweep_parameter`) to identify a promising region or performance plateau, followed by a focused search using methods such as:

    * GridSearchCV
    * RandomizedSearchCV
    * Another reasonable hyperparameter search method

* Choose hyperparameter values based on the validation results. If several nearby values produce similar validation performance (a performance plateau), prefer **values near the beginning of the plateau,** since they often produce simpler models with nearly identical predictive performance.
* Retrain the model using those hyperparameters.
* Evaluate the tuned model using repeated 5-fold cross-validation (5 repeats). 
* Report the validation performance (**mean** and **standard deviation** of the CV MAE).


In [ ]:
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor

from sklearn.model_selection import (
    GridSearchCV,
    RandomizedSearchCV,
    KFold,
    RepeatedKFold,
    cross_val_score
)

In [ ]:
# Used during hyperparameter tuning
tuning_cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


repeated_cv = RepeatedKFold(
    n_splits=5,
    n_repeats=5,
    random_state=42
)

In [ ]:
X_train_outliers
y_train_outliers

NameError: name 'X_train_outliers' is not defined

In [ ]:
print("X shape:", X_train_outliers.shape)
print("y shape:", y_train_outliers.shape)
print("Missing X values:", X_train_outliers.isna().sum().sum())
print("Missing y values:", y_train_outliers.isna().sum())

NameError: name 'X_train_outliers' is not defined

In [ ]:
#Tuning Linear Regression
linear_param_grid = {
    "fit_intercept": [True, False]
}

linear_search = GridSearchCV(
    estimator=LinearRegression(),
    param_grid=linear_param_grid,
    scoring="neg_mean_absolute_error",
    cv=tuning_cv,
    n_jobs=1,
    return_train_score=True
)

linear_search.fit(
    X_train_outliers,
    y_train_outliers
)

print("Best Linear Regression parameters:")
print(linear_search.best_params_)

print(
    "Best tuning CV MAE:",
    -linear_search.best_score_
)

{'ridge__alpha': 0.1}
199409.781314763


In [ ]:
linear_tuning_results = pd.DataFrame(
    linear_search.cv_results_
)[
    [
        "param_fit_intercept",
        "mean_test_score",
        "std_test_score"
    ]
].copy()

linear_tuning_results["Mean CV MAE"] = (
    -linear_tuning_results["mean_test_score"]
)

linear_tuning_results["Std CV MAE"] = (
    linear_tuning_results["std_test_score"]
)

linear_tuning_results[
    [
        "param_fit_intercept",
        "Mean CV MAE",
        "Std CV MAE"
    ]
].sort_values("Mean CV MAE")

In [ ]:
#Tuning Ridge Regression
ridge_param_grid = {
    "alpha": [
        0.01,
        0.1,
        1,
        10,
        25,
        50,
        100,
        250,
        500
    ]
}

ridge_search = GridSearchCV(
    estimator=Ridge(),
    param_grid=ridge_param_grid,
    scoring="neg_mean_absolute_error",
    cv=tuning_cv,
    n_jobs=1,
    return_train_score=True
)

ridge_search.fit(
    X_train_outliers,
    y_train_outliers
)

print("Best Ridge parameters:")
print(ridge_search.best_params_)

print(
    "Best tuning CV MAE:",
    -ridge_search.best_score_
)

In [ ]:
ridge_tuning_results = pd.DataFrame(
    ridge_search.cv_results_
)[
    [
        "param_alpha",
        "mean_test_score",
        "std_test_score"
    ]
].copy()

ridge_tuning_results["Mean CV MAE"] = (
    -ridge_tuning_results["mean_test_score"]
)

ridge_tuning_results["Std CV MAE"] = (
    ridge_tuning_results["std_test_score"]
)

ridge_tuning_results[
    [
        "param_alpha",
        "Mean CV MAE",
        "Std CV MAE"
    ]
].sort_values("param_alpha")

In [ ]:
best_ridge = ridge_search.best_estimator_

print(best_ridge)

In [ ]:
#Tuning Random Forest
rf_param_distributions = {
    "n_estimators": [100, 150, 200],
    "max_depth": [10, 15, 20, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", 0.5]
}

In [ ]:
#Tuning Random Forest

rf_search = RandomizedSearchCV(
    estimator=RandomForestRegressor(
        random_state=42,
        n_jobs=-1
    ),
    param_distributions=rf_param_distributions,
    n_iter=8,
    scoring="neg_mean_absolute_error",
    cv=tuning_cv,
    random_state=42,
    n_jobs=1,
    verbose=2,
    return_train_score=False
)

rf_search.fit(
    X_train_outliers,
    y_train_outliers
)

print("Best Random Forest parameters:")
print(rf_search.best_params_)

print(
    "Best tuning CV MAE:",
    -rf_search.best_score_
)

Fitting 5 folds for each of 8 candidates, totalling 40 fits


[CV] END max_depth=20, max_features=sqrt, min_samples_split=2, n_estimators=100; total time=  11.2s
[CV] END max_depth=20, max_features=sqrt, min_samples_split=2, n_estimators=100; total time=  11.2s
[CV] END max_depth=20, max_features=sqrt, min_samples_split=2, n_estimators=100; total time=  11.1s
[CV] END max_depth=20, max_features=sqrt, min_samples_split=2, n_estimators=100; total time=  11.3s
[CV] END max_depth=20, max_features=sqrt, min_samples_split=2, n_estimators=100; total time=  10.8s
[CV] END max_depth=None, max_features=sqrt, min_samples_split=2, n_estimators=100; total time=  16.3s
[CV] END max_depth=None, max_features=sqrt, min_samples_split=2, n_estimators=100; total time=  16.4s
[CV] END max_depth=None, max_features=sqrt, min_samples_split=2, n_estimators=100; total time=  15.3s
[CV] END max_depth=None, max_features=sqrt, min_samples_split=2, n_estimators=100; total time=  14.9s
[CV] END max_depth=None, max_features=sqrt, min_samples_split=2, n_estimators=100; total tim

In [ ]:
rf_tuning_results = pd.DataFrame(
    rf_search.cv_results_
)

rf_tuning_summary = rf_tuning_results[
    [
        "param_n_estimators",
        "param_max_depth",
        "param_min_samples_split",
        "param_min_samples_leaf",
        "param_max_features",
        "mean_test_score",
        "std_test_score"
    ]
].copy()

rf_tuning_summary["Mean CV MAE"] = (
    -rf_tuning_summary["mean_test_score"]
)

rf_tuning_summary["Std CV MAE"] = (
    rf_tuning_summary["std_test_score"]
)

rf_tuning_summary[
    [
        "param_n_estimators",
        "param_max_depth",
        "param_min_samples_split",
        "param_min_samples_leaf",
        "param_max_features",
        "Mean CV MAE",
        "Std CV MAE"
    ]
].sort_values("Mean CV MAE")

In [ ]:
best_rf = rf_search.best_estimator_

print(best_rf)

In [ ]:
best_linear = linear_search.best_estimator_

linear_scores = -cross_val_score(
    best_linear,
    X_train_outliers,
    y_train_outliers,
    scoring="neg_mean_absolute_error",
    cv=repeated_cv,
    n_jobs=1
)

print("Tuned Linear Regression")
print(f"Mean CV MAE: {linear_scores.mean():,.2f}")
print(f"Std CV MAE: {linear_scores.std():,.2f}")

In [ ]:
ridge_scores = -cross_val_score(
    best_ridge,
    X_train_outliers,
    y_train_outliers,
    scoring="neg_mean_absolute_error",
    cv=repeated_cv,
    n_jobs=1
)

print("Tuned Ridge Regression")
print(f"Mean CV MAE: {ridge_scores.mean():,.2f}")
print(f"Std CV MAE: {ridge_scores.std():,.2f}")

In [ ]:
rf_scores = -cross_val_score(
    best_rf,
    X_train_outliers,
    y_train_outliers,
    scoring="neg_mean_absolute_error",
    cv=repeated_cv,
    n_jobs=1,
    verbose=2
)

print("Tuned Random Forest")
print(f"Mean CV MAE: {rf_scores.mean():,.2f}")
print(f"Std CV MAE: {rf_scores.std():,.2f}")

In [ ]:
tuned_results = pd.DataFrame({
    "Model": [
        "Tuned Linear Regression",
        "Tuned Ridge Regression",
        "Tuned Random Forest"
    ],
    "Selected Hyperparameters": [
        str(linear_search.best_params_),
        str(ridge_search.best_params_),
        str(rf_search.best_params_)
    ],
    "Mean CV MAE": [
        linear_scores.mean(),
        ridge_scores.mean(),
        rf_scores.mean()
    ],
    "Std CV MAE": [
        linear_scores.std(),
        ridge_scores.std(),
        rf_scores.std()
    ]
})

tuned_results = tuned_results.sort_values(
    "Mean CV MAE"
).reset_index(drop=True)

tuned_results

In [ ]:
display_results = tuned_results.copy()

display_results["Mean CV MAE"] = (
    display_results["Mean CV MAE"]
    .map(lambda x: f"${x:,.2f}")
)

display_results["Std CV MAE"] = (
    display_results["Std CV MAE"]
    .map(lambda x: f"${x:,.2f}")
)

display_results

### 4.B Discussion

Answer the following questions.

#### 4.B.1

Which hyperparameters had the greatest impact on model performance? Briefly explain.

> Replace this text with your answer.

#### 4.B.2

Did hyperparameter tuning substantially improve the performance of all three models, or only some of them?

> Replace this text with your answer.

#### 4.B.3

Which tuning method(s) did you use for each model? Briefly explain why you chose those methods.

> Replace this text with your answer.

#### 4.B.4

After tuning, how did the relative performance of your three models change? Did tuning affect which model appeared to perform best?

> Replace this text with your answer.

## Part 5: Final Model and Workflow Assessment [14 pts]

### 5.A Coding

Using the work completed in **Parts 1–4**:

Select your **best-performing model** and prepare your final modeling pipeline.

Your pipeline should include all preprocessing, feature engineering, feature selection, and hyperparameter tuning decisions that you chose to retain.

Evaluate your final model by:

* Training on the complete training dataset.
* Reporting the **mean** and **standard deviation** of the repeated cross-validation MAE.
* Evaluating the model on the held-out test set.
* Reporting the final test MAE.

In [ ]:
# Add as many code cells as needed.

### 5.B Discussion

Answer the following questions.

#### 5.B.1

Compare the performance of your final model with its original baseline from **Part 1**. Which changes contributed the most to the improvement?

> Replace this text with your answer.

#### 5.B.2

Looking back at the hypotheses you proposed in **Milestone 1**, which were supported by your experimental results? Were any hypotheses disproved?

> Replace this text with your answer.

#### 5.B.3

Why did you select this model as your final model? Discuss both its predictive performance and any other considerations (such as stability, simplicity, or interpretability).

> Replace this text with your answer.

#### 5.B.4

What did you learn about your dataset and the machine learning process through this end-to-end modeling workflow? If you had additional time, what would you investigate next?

> Replace this text with your answer.